<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/ml/notebooks/c5_l3.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C5-L3 · Validación walk-forward
3 folds temporales con embargo: entrena en pasado, evalúa en el bloque siguiente.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/ml/data/c5_l3.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c5_l3.csv'), Path('data/c5_l3.csv'), Path('c5_l3.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
# Features + etiqueta (retorno siguiente)
for k in range(1, 6):
    df[f'lag_{k}'] = df['close'].shift(k)
df['ret_next'] = df['close'].pct_change().shift(-1)
feat = [f'lag_{k}' for k in range(1, 6)]
data = df.dropna().reset_index(drop=True)
print('filas utiles:', len(data))
assert len(data) > 60

In [ ]:
# Walk-forward expanding: 3 folds, embargo de 5 entre train y test
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
N, FOLDS, EMB = len(data), 3, 5
edges = [int(N*(0.5 + 0.5*i/FOLDS)) for i in range(FOLDS+1)]  # [50%, 66%, 83%, 100%]
print('cortes:', edges)
rmses, hits = [], []
for f in range(FOLDS):
    tr_end, te_ini, te_end = edges[f]-EMB, edges[f], edges[f+1]
    Xtr, ytr = data[feat].values[:tr_end], data['ret_next'].values[:tr_end]
    Xte, yte = data[feat].values[te_ini:te_end], data['ret_next'].values[te_ini:te_end]
    assert tr_end <= te_ini, 'el embargo debe separar train y test'
    m = Ridge().fit(Xtr, ytr)
    p = m.predict(Xte)
    rmses.append(mean_squared_error(yte, p) ** 0.5)
    hits.append(float(((p > 0) == (yte > 0)).mean()))
    print(f'fold {f+1}: train<={tr_end} test=[{te_ini},{te_end}) RMSE={rmses[-1]:.6f} hit={hits[-1]:.3f}')
assert len(rmses) == 3

In [ ]:
# Reporte OOS honesto: promedio +- dispersion (no el mejor pliegue)
rmses = np.array(rmses); hits = np.array(hits)
print(f'RMSE OOS = {rmses.mean():.6f} +- {rmses.std():.6f}')
print(f'hit-rate OOS = {hits.mean():.3f} +- {hits.std():.3f}')
# k-fold aleatorio (tramposo) como contraste
from sklearn.model_selection import cross_val_score
cf = cross_val_score(Ridge(), data[feat].values, data['ret_next'].values, cv=3).mean()
print(f'R2 k-fold aleatorio (referencia, mezcla futuro)={cf:.4f}')

In [ ]:
# Chequeo automatico L3
assert len(rmses) == 3 and len(hits) == 3
assert all(np.isfinite(rmses)) and all(0 <= h <= 1 for h in hits)
assert all(r < 0.2 for r in rmses), 'RMSE OOS debe ser razonable en retornos diarios'
print('OK L3: walk-forward 3 folds verificado')